## Lets match licenses to pins for the city of Savannah

### Normalize Savannah license addresses (no geocoding yet)

Run the next cell from the project root or the notebooks folder. It preserves all source columns and rows and writes `data/savannah_licenses/Licensed Short-Term Rental Properties.matched.csv`. Re-running replaces only that generated file.

Uses [USPS common street suffix abbreviations](https://pe.usps.com/text/pub28/28apc_002.htm), uppercase, whitespace/punctuation cleanup, and directional prefixes/suffixes. CROSSING stays XING, CROSSROAD becomes XRD, and CROSSROADS becomes XRDS. The local FACTORS WK spelling becomes FACTORS WALK. Street-name words are preserved rather than blindly abbreviated.

`Address_Match_Key` is the street address without unit or locality, for later parcel matching; use city/ZIP as additional constraints when comparing across jurisdictions. `Unit_Match_Key` includes the unit to distinguish rentals in the same building. Embedded units are retained separately, and conflicts with the unit column are flagged (the explicit unit column takes precedence). Missing locality information is flagged rather than guessed. Normalized does not mean verified, licensed at that location, or geocoded. No rows are merged, and no coordinates or parcel matches are invented.

`Parcel_PIN` copies the source `APN` as text (leading zeros preserved); it is a supplied identifier, not a newly verified parcel match.


In [1]:
from pathlib import Path
import csv
import re
import unicodedata
from collections import Counter

# Self-contained, offline normalization; no geocoding or parcel matches yet.
# Common suffix forms: https://pe.usps.com/text/pub28/28apc_002.htm
# Standard suffixes first; do not conflate CROSSING with CROSSROAD(S).
# WK is a local shorthand present in three FACTORS WK addresses; normalize to WALK.
SUFFIX_GROUPS = '''
ALY ALLEY ALLEE ALLY
ANX ANEX ANNEX ANNX
ARC ARCADE
AVE AVENUE AV AVEN AVENU AVN AVNUE
BYU BAYOU BAYOO
BCH BEACH
BND BEND
BLF BLUFF BLUF
BLFS BLUFFS
BTM BOTTOM BOT BOTTM
BLVD BOULEVARD BOUL BOULV
BR BRANCH BRNCH
BRG BRIDGE BRDGE
BRK BROOK
BRKS BROOKS
BG BURG
BGS BURGS
BYP BYPASS BYPA BYPAS BYPS
CP CAMP CMP
CYN CANYON CANYN CNYN
CPE CAPE
CSWY CAUSEWAY CAUSWA
CTR CENTER CENTRE CEN CENT CENTR CNTER CNTR
CTRS CENTERS
CIR CIRCLE CIRC CIRCL CRCL CRCLE
CIRS CIRCLES
CLF CLIFF
CLFS CLIFFS
CLB CLUB
CMN COMMON
CMNS COMMONS
COR CORNER
CORS CORNERS
CRSE COURSE
CT COURT
CTS COURTS
CV COVE
CVS COVES
CRK CREEK
CRES CRESCENT CRSENT CRSNT
CRST CREST
XING CROSSING CRSSNG
XRD CROSSROAD
XRDS CROSSROADS
CURV CURVE
DL DALE
DM DAM
DV DIVIDE DIV DVD
DR DRIVE DRIV DRV
DRS DRIVES
EST ESTATE
ESTS ESTATES
EXPY EXPRESSWAY EXP EXPR EXPRESS EXPW
EXT EXTENSION EXTN EXTNSN
EXTS EXTENSIONS
FALL FALL
FLS FALLS
FRY FERRY FRRY
FLD FIELD
FLDS FIELDS
FLT FLAT
FLTS FLATS
FRD FORD
FRDS FORDS
FRST FOREST FORESTS
FRG FORGE FORG
FRGS FORGES
FRK FORK
FRKS FORKS
FT FORT FRT
FWY FREEWAY FREEWY FRWAY FRWY
GDN GARDEN GARDN GRDEN GRDN
GDNS GARDENS GRDNS
GTWY GATEWAY GATEWY GATWAY GTWAY
GLN GLEN
GLNS GLENS
GRN GREEN
GRNS GREENS
GRV GROVE GROV
GRVS GROVES
HBR HARBOR HARBOUR HARB HARBR HRBOR
HBRS HARBORS
HVN HAVEN
HTS HEIGHTS HT
HWY HIGHWAY HIGHWY HIWAY HIWY HWAY
HL HILL
HLS HILLS
HOLW HOLLOW HLLW HOLLOWS HOLWS
INLT INLET
IS ISLAND ISLND
ISS ISLANDS ISLNDS
ISLE ISLES
JCT JUNCTION JCTION JCTN JUNCTN JUNCTON
JCTS JUNCTIONS JCTNS
KY KEY
KYS KEYS
KNL KNOLL KNOL
KNLS KNOLLS
LK LAKE
LKS LAKES
LAND LAND
LNDG LANDING LNDNG
LN LANE
LGT LIGHT
LGTS LIGHTS
LF LOAF
LCK LOCK
LCKS LOCKS
LDG LODGE LDGE LODG
LOOP LOOPS
MALL MALL
MNR MANOR
MNRS MANORS
MDW MEADOW
MDWS MEADOWS MEDOWS
MEWS MEWS
ML MILL
MLS MILLS
MSN MISSION MISSN MSSN
MTWY MOTORWAY
MT MOUNT MNT
MTN MOUNTAIN MNTAIN MNTN MOUNTIN MTIN
MTNS MOUNTAINS MNTNS
NCK NECK
ORCH ORCHARD ORCHRD
OVAL OVL
OPAS OVERPASS
PARK PARKS PRK
PKWY PARKWAY PARKWY PKWAY PKY PARKWAYS PKWYS
PASS PASS
PSGE PASSAGE
PATH PATHS
PIKE PIKES
PNE PINE
PNES PINES
PL PLACE
PLN PLAIN
PLNS PLAINS
PLZ PLAZA PLZA
PT POINT
PTS POINTS
PRT PORT
PRTS PORTS
PR PRAIRIE PRR
RADL RADIAL RAD RADIEL
RAMP RAMP
RNCH RANCH RANCHES RNCHS
RPD RAPID
RPDS RAPIDS
RST REST
RDG RIDGE RDGE
RDGS RIDGES
RIV RIVER RVR RIVR
RD ROAD
RDS ROADS
RTE ROUTE
ROW ROW
RUE RUE
RUN RUN
SHL SHOAL
SHLS SHOALS
SHR SHORE SHOAR
SHRS SHORES SHOARS
SKWY SKYWAY
SPG SPRING SPNG SPRNG
SPGS SPRINGS SPNGS SPRNGS
SPUR SPURS
SQ SQUARE SQR SQRE SQU
SQS SQUARES SQRS
STA STATION STATN STN
STRA STRAVENUE STRAV STRAVEN STRAVN STRVN STRVNUE
STRM STREAM STREME
ST STREET STR STRT
STS STREETS
SMT SUMMIT SUMIT SUMITT
TER TERRACE TERR
TRWY THROUGHWAY
TRCE TRACE TRACES
TRAK TRACK TRACKS TRK TRKS
TRFY TRAFFICWAY
TRL TRAIL TRAILS TRLS
TRLR TRAILER TRAILERS TRLRS
TUNL TUNNEL TUNEL TUNLS TUNNELS TUNNL
TPKE TURNPIKE TRNPK TURNPK
UPAS UNDERPASS
UN UNION
UNS UNIONS
VLY VALLEY VALLY VLLY
VLYS VALLEYS
VIA VIADUCT VDCT VIADCT
VW VIEW
VWS VIEWS
VLG VILLAGE VILL VILLAG VILLG VILLIAGE
VLGS VILLAGES
VL VILLE
VIS VISTA VIST VST VSTA
WALK WALKS WK
WALL WALL
WAY WY
WAYS WAYS
WL WELL
WLS WELLS
'''
SUFFIXES = {alias: words[0] for line in SUFFIX_GROUPS.strip().splitlines()
            for words in [line.split()] for alias in words}
DIRECTIONS = dict(NORTH='N', SOUTH='S', EAST='E', WEST='W',
                  NORTHEAST='NE', NORTHWEST='NW', SOUTHEAST='SE', SOUTHWEST='SW')
DIRECTIONS.update({v: v for v in list(DIRECTIONS.values())})

def clean(value):
    text = unicodedata.normalize('NFKC', str(value or '')).upper()
    text = text.replace('½', ' 1/2').replace('⁄', '/')
    text = re.sub(r"[.'’]", '', text)
    return re.sub(r'\s+', ' ', text.replace(',', ' ')).strip()

def normalize_unit(value):
    return re.sub(r'^(?:APARTMENT|APT|UNIT|SUITE|STE|#)\s*', '', clean(value)).strip()

def normalize_address(raw_address, raw_unit=''):
    text = clean(raw_address)
    notes = []
    # Preserve supplied ZIP; never guess/correct a ZIP from the street name.
    locality = re.search(r'\b(?:(SAVANNAH)\s+)?(GA|GEORGIA)\s+(\d{5}(?:-\d{4})?)$', text)
    city = state = zipcode = ''
    if locality:
        city, state, zipcode = locality.group(1) or '', 'GA', locality.group(3)
        text = text[:locality.start()].strip()
        if not city:
            notes.append('Missing city in source')
    else:
        notes.append('Unrecognized city/state/ZIP ending')
    embedded_unit = ''
    unit_match = re.search(r'\s+(?:APARTMENT|APT|UNIT|SUITE|STE|#)\s*(.+)$', text)
    if unit_match:
        embedded_unit = normalize_unit(unit_match.group(1))
        text = text[:unit_match.start()]
    tokens = text.split()
    # Locate the last suffix, never replace words inside the street name
    # (e.g. PARK AVE or ST JULIAN ST). Keep house fractions and hyphens.
    candidates = [i for i, token in enumerate(tokens) if i >= 2 and token in SUFFIXES]
    if candidates:
        end = candidates[-1]
        tail = tokens[end+1:]
        postdirection = ''
        if tail and ' '.join(tail) in DIRECTIONS:
            postdirection = DIRECTIONS[' '.join(tail)]
        elif tail:
            tail_unit = normalize_unit(' '.join(tail))
            if embedded_unit and embedded_unit != tail_unit:
                notes.append('Multiple embedded unit values')
            else:
                embedded_unit = tail_unit
        tokens = tokens[:end] + [SUFFIXES[tokens[end]]]
        # Directional prefixes only, including NORTH EAST -> NE.
        start = 2 if len(tokens) > 2 and re.fullmatch(r'\d+/\d+', tokens[1]) else 1
        if len(tokens) > start + 3 and ''.join(tokens[start:start+2]) in DIRECTIONS:
            tokens[start:start+2] = [DIRECTIONS[''.join(tokens[start:start+2])]]
        elif len(tokens) > start + 2 and tokens[start] in DIRECTIONS:
            tokens[start] = DIRECTIONS[tokens[start]]
        if postdirection:
            tokens.append(postdirection)
    else:
        notes.append('Unrecognized street suffix; review street/unit split')
    street = ' '.join(tokens)
    if not re.match(r'^\d+[A-Z]?(?:-\d+[A-Z]?)?\s+\S', street):
        notes.append('Missing or unusual house number')
    unit = normalize_unit(raw_unit)
    if unit and embedded_unit and unit != embedded_unit:
        notes.append('Unit column conflicts with embedded unit')
    unit = unit or embedded_unit
    street_unit = street + (f' UNIT {unit}' if unit else '')
    full = ', '.join(part for part in [street_unit, city, ' '.join(filter(None, [state, zipcode]))] if part)
    return dict(Normalized_Street_Address=street, Normalized_Unit=unit,
                Embedded_Unit=embedded_unit, Normalized_City=city, Normalized_State=state,
                Normalized_ZIP=zipcode, Normalized_Full_Address=full,
                Address_Match_Key=street, Unit_Match_Key=street_unit,
                Normalization_Status='Needs review' if notes else 'Normalized',
                Normalization_Notes='; '.join(notes))

# Focused checks protect street names, fractions, directionals, and unit identity.
assert normalize_address('501 East McDonough Street, Savannah GA 31401')['Address_Match_Key'] == '501 E MCDONOUGH ST'
assert normalize_address('12 West Park Avenue Savannah GA 31401')['Address_Match_Key'] == '12 W PARK AVE'
assert normalize_address('12 EAST SAINT JULIAN ST SAVANNAH GA 31401')['Address_Match_Key'] == '12 E SAINT JULIAN ST'
assert normalize_address('12 North East Oak Cove Savannah GA 31401')['Address_Match_Key'] == '12 NE OAK CV'
assert normalize_address('12 Oak Crossroads Savannah GA 31401')['Address_Match_Key'] == '12 OAK XRDS'
assert normalize_address('12 1/2 E Jones St # A Savannah GA 31401')['Unit_Match_Key'] == '12 1/2 E JONES ST UNIT A'
assert normalize_address('12 E Jones St A Savannah GA 31401', 'B')['Normalization_Status'] == 'Needs review'

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "county_licenses_map.html").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the stvr_map project.")
source = root / 'data' / 'savannah_licenses' / 'Licensed Short-Term Rental Properties.csv'
output = source.with_name(source.stem + '.matched.csv')
with source.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    original_fields = reader.fieldnames
    rows = list(reader)
# APN is the source property/parcel identifier; keep it as text, including leading zeros.
normalized_rows = [{**row, 'Parcel_PIN': row['APN'].strip(),
                    **normalize_address(row['Address'], row['Unit Number'])} for row in rows]
extra_fields = ['Parcel_PIN'] + list(normalize_address('', ''))
with output.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=original_fields + extra_fields)
    writer.writeheader()
    writer.writerows(normalized_rows)
# Verify row order, original data, and identifiers survive the CSV round trip.
with output.open(encoding='utf-8-sig', newline='') as handle:
    saved = list(csv.DictReader(handle))
assert len(saved) == len(rows)
assert all(after['Parcel_PIN'] == before['APN'].strip() for before, after in zip(rows, saved))
assert all({key: after[key] for key in original_fields} == before for before, after in zip(rows, saved))
assert all(normalize_address(r['Normalized_Full_Address'])['Address_Match_Key'] == r['Address_Match_Key'] for r in saved)
print(f'Created: {output.relative_to(root)}')
print(f'Rows preserved: {len(saved):,}')
print(f'Status counts: {dict(Counter(r["Normalization_Status"] for r in saved))}')
print(f'Unique street matching keys: {len({r["Address_Match_Key"] for r in saved}):,}')
print('Normalization only: no parcel matches or coordinates have been assigned.')
for row in saved:
    if row['Normalization_Status'] == 'Needs review':
        print(row['Permit Number'], row['Address'], '|', row['Normalization_Notes'])


Created: data\savannah_licenses\Licensed Short-Term Rental Properties.matched.csv
Rows preserved: 1,679
Status counts: {'Normalized': 1674, 'Needs review': 5}
Unique street matching keys: 1,000
Normalization only: no parcel matches or coordinates have been assigned.
SVR-01038 101 E GASTON ST 6A, SAVANNAH, GA, 31401 | Unit column conflicts with embedded unit
SVR-01041 101 E GASTON ST 4A, SAVANNAH, GA, 31401 | Unit column conflicts with embedded unit
SVR-00026 217 ABERCORN ST # A3 SAVANNAH GA 31401 | Unit column conflicts with embedded unit
SVR-03110 404 UPPER EAST AVE GA 31401 | Missing city in source
SVR-03215 105 W OGLETHORPE AVE 1, SAVANNAH, GA, 31401 | Unit column conflicts with embedded unit
